<a href="https://colab.research.google.com/github/chiragbulbule/VaultLLM/blob/main/TenSEAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install tenseal
import tenseal as ts

In [5]:
context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=8192,
    coeff_mod_bit_sizes=[60, 40, 40, 60]
)
context.global_scale = 2**40
context.generate_galois_keys()
plain_number = [7.5]
encrypted = ts.ckks_vector(context, plain_number)
encrypted_result = encrypted * 2
result = encrypted_result.decrypt()
print(f"Original: 7.5 × 2 = 15.0")
print(f"FHE Result: {result[0]:.4f}")

Original: 7.5 × 2 = 15.0
FHE Result: 15.0000


In [6]:
import tenseal as ts

# Same setup as before
context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=8192,
    coeff_mod_bit_sizes=[60, 40, 40, 60]
)
context.global_scale = 2**40
context.generate_galois_keys()

# A vector (pretend this is a tiny AI embedding)
plain_vector = [1.0, 2.0, 3.0, 4.0, 5.0]

# Encrypt the whole vector at once
encrypted_vector = ts.ckks_vector(context, plain_vector)

# Multiply every element by 2, while encrypted
encrypted_result = encrypted_vector * 2

# Decrypt
result = encrypted_result.decrypt()

print("Original:  ", plain_vector)
print("Expected:  ", [x * 2 for x in plain_vector])
print("FHE Result:", [round(x, 4) for x in result])

Original:   [1.0, 2.0, 3.0, 4.0, 5.0]
Expected:   [2.0, 4.0, 6.0, 8.0, 10.0]
FHE Result: [2.0, 4.0, 6.0, 8.0, 10.0]


In [17]:
import tenseal as ts

context = ts.context(
    ts.SCHEME_TYPE.CKKS,
poly_modulus_degree=16384,
coeff_mod_bit_sizes=[60, 40, 40, 40, 40, 40, 40, 60]  # 6 middle levels
)
context.global_scale = 2**40
context.generate_galois_keys()

plain_vector = [1.0, 2.0, 3.0, 4.0, 5.0]
encrypted_vector = ts.ckks_vector(context, plain_vector)

# Multiply but rescale between each operation
result = encrypted_vector * 2 * 2 * 2 * 2 * 2 * 2 * 2 * 2 * 2 * 2

decrypted = result.decrypt()

print("Original:       ", plain_vector)
print("Expected (×32): ", [x * 32 for x in plain_vector])
print("FHE Result:     ", [round(x, 4) for x in decrypted])

ValueError: scale out of bounds

In [23]:
import tenseal as ts

# ============================================================
# SETUP (Cryptographer's job — your code)
# ============================================================
context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=16384,
    coeff_mod_bit_sizes=[60, 40, 40, 40, 40, 40, 40, 60]
)
context.global_scale = 2**40        # ← this line is missing in your cell
context.generate_galois_keys()
# ============================================================
# CLIENT SIDE (User's machine)
# ============================================================

# Step 1: User types a query
user_query = "The machine is overheating"
print(f"User types: '{user_query}'")

# Step 2: Convert to numbers (pretend embedding, AI engineer's job)
# In real project this would be a 768-dimension vector from a real model
# For now we simulate it with small numbers
plain_embedding = [0.23, -0.87, 0.45, 1.02, -0.33]
print(f"Converted to vector: {plain_embedding}")

# Step 3: Encrypt it (YOUR job)
encrypted_embedding = ts.ckks_vector(context, plain_embedding)
print(f"\nEncrypted (what server sees): {encrypted_embedding}")

# ============================================================
# SERVER SIDE (Blind — never sees real data)
# ============================================================
print("\n--- SERVER RECEIVES ENCRYPTED BLOB ---")
print("Server has NO idea what the original query was.")

# Step 4: Server does AI math on encrypted data
# In real project this would be encrypted matrix multiplication
# For now we simulate a simple weighted sum (like a tiny neural network layer)
weights = [0.5, 0.3, 0.8, 0.2, 0.6]  # pretend model weights
encrypted_result = encrypted_embedding.dot(weights)
print("Server computed result on ciphertext — still blind!")

# ============================================================
# BACK TO CLIENT SIDE
# ============================================================
print("\n--- RESULT RETURNS TO USER ---")

# Step 5: User decrypts
decrypted_result = encrypted_result.decrypt()
score = decrypted_result[0]
print(f"Decrypted score: {round(score, 4)}")

# Step 6: Interpret result (researcher/AI engineer's job)
if score > 0:
    print("Classification: ⚠️  ALERT — Negative signal detected")
else:
    print("Classification: ✅  OK — Normal signal")

User types: 'The machine is overheating'
Converted to vector: [0.23, -0.87, 0.45, 1.02, -0.33]

Encrypted (what server sees): <tenseal.tensors.ckksvector.CKKSVector object at 0x7fc218289850>

--- SERVER RECEIVES ENCRYPTED BLOB ---
Server has NO idea what the original query was.
Server computed result on ciphertext — still blind!

--- RESULT RETURNS TO USER ---
Decrypted score: 0.22
Classification: ⚠️  ALERT — Negative signal detected
